In [ ]:
!pip -q install langchain openai tiktoken chromadb pypdf sentence_transformers InstructorEmbedding faiss-cpu pinecone-client unstructured baichat-py

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 29.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 10.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 94.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 18.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.3/256.3 kB 34.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 68.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.1/179.1 kB 26.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 67.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.0/90.0 kB 10.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 7.6 MB/s eta 0:00

In [ ]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Pinecone
import pinecone
import os
from langchain.document_loaders import DirectoryLoader, TextLoader
from baichat_py import Completion
from langchain.text_splitter import RecursiveCharacterTextSplitter , CharacterTextSplitter

/usr/local/lib/python3.10/dist-packages/pinecone/index.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
os.environ['PINECONE_API_KEY']= "Enter you pincone Api key"
os.environ['PINECONE_ENV']= "us-west4-gcp-free"
os.environ['OPENAI_API_KEY']="Enter you open Api key"
pinecone.init(
    api_key=os.environ['PINECONE_API_KEY'],  # find at app.pinecone.io
    environment=os.environ['PINECONE_ENV']  # next to api key in console
)

In [ ]:
os.environ['PINECONE_INDEX_NAME'] = 'geetagpt'

In [ ]:
# from dotenv import load_env
os.environ['PINECONE_API_KEY']= "Enter you pincone Api key"
os.environ['PINECONE_ENV']= "us-west4-gcp-free"
os.environ['OPENAI_API_KEY']="Enter you open Api key"
from langchain.embeddings import HuggingFaceInstructEmbeddings

instructor_embeddings = HuggingFaceInstructEmbeddings(model_name="hkunlp/instructor-xl",
                                                      query_instruction="Represent the query for retrieval: ",
                                                      model_kwargs={"device": "cuda"})


load INSTRUCTOR_Transformer
max_seq_length  512


## build index

In [ ]:
my_loader = DirectoryLoader('./', glob='**/*.txt')
documents = my_loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 512, chunk_overlap = 20)
docs = text_splitter.split_documents(documents)

In [ ]:

geeta_txt = TextLoader('/content/Geeta-GPT/geeta.txt').load()
docs += CharacterTextSplitter(chunk_size = 512, chunk_overlap = 20).split_documents(geeta_txt)

In [ ]:
documents

In [ ]:
print(len(docs))

7697


Document(page_content='Bhagavad-gita As It Is is also published in a hardcover edition by the Macmillan Company.\n\nMacmillan Publishing Co., Inc.\n\nCollier\n\n\n\nMacmillan Canada Ltd.\n\nPrinted in the United States of America\n\nTo\n\nSRILA baladeva vidyAbhusana\n\nwho presented so nicely\n\nthe "Govinda\n\n\n\nbhasya " commentary\n\non\n\nVedanta philosophy\n\nTABLE OF CONTENTS\n\nForeword\n\nPreface\n\nIntroduction\n\nCHAPTER ONE\n\nObserving the Armies on the Battlefield of Kuruksetra\n\nCHAPTER TWO\n\nContents of the Gita Summarized\n\nCHAPTER THREE\n\nKarma', metadata={'source': 'geeta.txt'})

In [ ]:
docs[4].page_content = 'CHAPTER ONE\n\nObserving the Armies on the Battlefield of Kuruksetra\n\nCHAPTER TWO\n\nContents of the Gita Summarized\n\nCHAPTER THREE\n\nKarma'

In [ ]:
docs = docs[4:]

In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings

# instructor_embeddings = HuggingFaceInstructEmbeddings(model_name="intfloat/e5-large-v2",
#                                                       model_kwargs={"device": "cuda"})

In [ ]:
pinecone.init(
    api_key=os.environ['PINECONE_API_KEY'],  # find at app.pinecone.io
    environment=os.environ['PINECONE_ENV']  # next to api key in console
)

In [ ]:
def set_env(env_name,val):
    os.environ[env_name] = val

In [ ]:
set_env('PINECONE_INDEX_NAME','geetagpt')

In [ ]:

pinecone.create_index("geetagpt", dimension=768)

In [ ]:
!git clone https://github.com/alt-coder/Geeta-GPT

Cloning into 'Geeta-GPT'...
remote: Enumerating objects: 669, done.
remote: Counting objects: 100% (669/669), done.
remote: Compressing objects: 100% (665/665), done.
remote: Total 669 (delta 0), reused 666 (delta 0), pack-reused 0
Receiving objects: 100% (669/669), 652.30 KiB | 17.63 MiB/s, done.


In [ ]:
docsearch = Pinecone.from_documents(docs, instructor_embeddings, index_name=os.environ['PINECONE_INDEX_NAME'])

In [ ]:
!find geeta.txt

find: ‘geeta.txt’: No such file or directory


#Load db

In [ ]:
docsearch =Pinecone.from_existing_index(index_name=os.environ['PINECONE_INDEX_NAME'],embedding=instructor_embeddings)

In [ ]:
# docsearch = Pinecone.from_existing_index(os.environ['PINECONE_INDEX_NAME'], instructor_embeddings)
query = "How to live Happy?"
docs = docsearch.similarity_search(query,)
from langchain.chains.qa_with_sources import load_qa_with_sources_chain
from langchain.llms import OpenAI

## Query LLM

In [ ]:
from typing import Any, List, Mapping, Optional
from langchain.chains.qa_with_sources import load_qa_with_sources_chain
from langchain.llms import OpenAI
from langchain.callbacks.manager import CallbackManagerForLLMRun
from langchain.llms.base import LLM

In [ ]:
def get_output(prompt):
      # print(prompt)
      from baichat_py import Completion
      try:
        tempstr=''
        print()
        for token in Completion.create(prompt):
            tempstr+=token
            print(token)
      except:
        tempstr='Please try again.'
      return tempstr

In [ ]:
print(get_output('Hi how are you'))

In [ ]:
class CustomLLM(LLM):
    def get_output(self,prompt):
      # print(prompt)
      from baichat_py import Completion
      try:
        tempstr=''
        for token in Completion.create(prompt):
            tempstr+=token
      except:
        tempstr='Please try again.'
      return tempstr

    @property
    def _llm_type(self) -> str:
        return "custom"

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
    ) -> str:
        if stop is not None:
            raise ValueError("stop kwargs are not permitted.")
        return self.get_output(prompt)

    @property
    def _identifying_params(self) -> Mapping[str, Any]:
        """Get the identifying parameters."""
        return {}

In [ ]:
# query = "What did dhratrastra ask about the battlefield?"
query = "did Pandavas and the sons of Dhrtarastra belong to the same family?"
docs = docsearch.similarity_search(query,k=5)

In [ ]:
docs

[Document(page_content="Both the Pandavas and the sons of Dhrtarastra belong to the same family, but Dhrtarastra's mind is disclosed herein. He deliberately claimed only his sons as Kurus, and he separated the sons of Pandu from the family heritage. One can thus understand the specific position of Dhrtarastra in his relationship with his nephews, the sons of Pandu. As in the paddy field the unnecessary plants are taken out, so it is expected from the very beginning of these topics that in the religious field of Kuruksetra where", metadata={'source': 'Geeta-GPT/geeta.txt'}),
 Document(page_content='Commentary: Lord Krishna took birth on the earth in the Vrishni dynasty as the son of Vasudev. Since no soul can excel the Lord, he is naturally the most glorious personality of the Vrishni dynasty. The Pandavas were the five sons of Pandu—Yudhishthir, Bheem, Arjun, Nakul, and Sahadev. Amongst them, Arjun was an archer par-excellence, and was a very intimate devotee of Shree Krishna. He looke

In [ ]:
model = CustomLLM()
sources_chain = load_qa_with_sources_chain(model, chain_type="refine")
result = sources_chain.run(input_documents=docs, question=query)
print(result)

In [ ]:
result

''

In [ ]:
'''the desire to attain success is natural to the soul, and it can be achieved in two realms - material and spiritual.
 Those who consider the world as a source of happiness strive for material advancement, while those who value spiritual wealth strive for it by
  rejecting material endeavors. However, if one wants to achieve true happiness, they need to balance their material and spiritual pursuits in life.
  Success and happiness mean different things to different people, and there is no one-size-fits-all answer.
  Therefore, to achieve success and happiness in life, one needs to understand their values and beliefs and determine what success and happiness mean
  to them personally. They should reflect on their goals and priorities and aim to balance their material and spiritual pursuits to lead a fulfilling life.

Sources:
- Geeta-GPT/Book/chapter2/66.txt
- Geeta-GPT/Book/chapter6/38.txt'''

'the desire to attain success is natural to the soul, and it can be achieved in two realms - material and spiritual.\n Those who consider the world as a source of happiness strive for material advancement, while those who value spiritual wealth strive for it by\n  rejecting material endeavors. However, if one wants to achieve true happiness, they need to balance their material and spiritual pursuits in life.\n  Success and happiness mean different things to different people, and there is no one-size-fits-all answer.\n  Therefore, to achieve success and happiness in life, one needs to understand their values and beliefs and determine what success and happiness mean\n  to them personally. They should reflect on their goals and priorities and aim to balance their material and spiritual pursuits to lead a fulfilling life.\n\nSources:\n- Geeta-GPT/Book/chapter2/66.txt\n- Geeta-GPT/Book/chapter6/38.txt'

In [ ]:
!pip install gradio_client

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.4/288.4 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 10.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.8/236.8 kB 28.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.9/129.9 kB 17.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.4 MB/s eta 0:00:00


In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)